# Beta hedging with Fonrex

This notebook builds an equally weighted portfolio of gold stocks and healthcare stocks, measures its alpha and beta against the S&P 500 (SPY), then builds a beta-hedged portfolio.

The daily prices come from **your own Fonrex instance** (`GET /eod/{ticker}`) instead of OpenBB.

**Before you run it**

1. A Fonrex instance is running (`docker compose up -d`).
2. The nine listings are in the catalogue, in US dollars (see the guide *Using Fonrex from Python and Jupyter*).
   Your Fonrex version accepts the `isin` parameter of `GET /eod`.
3. Set your key in the environment before starting Jupyter: `export FONREX_API_KEY=frx_live_...`. A read-only key is enough.

In [ ]:
%pip install statsmodels requests

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import statsmodels.api as sm

warnings.filterwarnings("ignore")

FONREX_URL = os.environ.get("FONREX_URL", "http://localhost:5000")
FONREX_API_KEY = os.environ["FONREX_API_KEY"]

`fonrex_prices()` replaces `obb.equity.price.historical(...).pivot(columns="symbol", values="close")`: one column per symbol, one row per trading session.

A ticker alone does not always designate one instrument: in the catalogue, `NEM` is also Nemetschek, `MRK` also Merck KGaA and `CDE` also City Developments. Each ticker comes with the **ISIN** of its instrument, and `currency="USD"` chooses its US listing.

In [ ]:
def fonrex_prices(instruments, start_date, end_date, currency="USD"):
    """Daily closing prices of several listings, one column per ticker.

    ``instruments`` maps each ticker to the ISIN of its instrument.
    """
    session = requests.Session()
    session.headers["X-API-KEY"] = FONREX_API_KEY
    closes = {}
    for ticker, isin in instruments.items():
        response = session.get(
            f"{FONREX_URL}/eod/{ticker}",
            params={"from": start_date, "to": end_date, "isin": isin, "currency": currency},
            timeout=120,  # the first call ingests the prices from Yahoo Finance
        )
        if response.status_code != 200:
            raise RuntimeError(f"{ticker}: {response.status_code} {response.text}")
        bars = pd.DataFrame(response.json()["data"])
        closes[ticker] = bars.set_index(pd.to_datetime(bars["Date"]))["Close"]
    data = pd.DataFrame(closes).sort_index(axis=1)
    data.index.name = "date"
    data.columns.name = "symbol"
    return data

Construct an equally weighted portfolio of gold stocks (NEM, RGLD, SSRM, CDE) and healthcare stocks (LLY, UNH, JNJ, MRK).

In [ ]:
instruments = {
    "NEM": "US6516391066",   # Newmont
    "RGLD": "US7802871084",  # Royal Gold
    "SSRM": "CA7847301032",  # SSR Mining
    "CDE": "US1921085049",   # Coeur Mining
    "LLY": "US5324571083",   # Eli Lilly
    "UNH": "US91324P1021",   # UnitedHealth
    "JNJ": "US4781601046",   # Johnson & Johnson
    "MRK": "US58933Y1055",   # Merck & Co
    "SPY": "US78462F1030",   # SPDR S&P 500 ETF
}
data = fonrex_prices(instruments, start_date="2020-01-01", end_date="2022-12-31")

In [ ]:
data

Extract the benchmark returns (SPY) and compute the daily returns.

In [ ]:
benchmark_returns = (
    data
    .pop("SPY")
    .pct_change()
    .dropna()
)
benchmark_returns

Compute the portfolio returns by summing the daily returns of each asset.

In [ ]:
portfolio_returns = (
    data
    .pct_change()
    .dropna()
    .sum(axis=1)
)
portfolio_returns.name = "portfolio"
portfolio_returns

Plot the portfolio returns against the benchmark returns.

In [ ]:
portfolio_returns.plot()
benchmark_returns.plot()
plt.ylabel("Daily Return")
plt.legend()

Regress the portfolio returns on the benchmark returns to estimate alpha and beta.

In [ ]:
X = benchmark_returns.values
Y = portfolio_returns.values


def linreg(x, y):
    # Add a column of 1s to fit alpha
    x = sm.add_constant(x)
    model = sm.OLS(y, x).fit()
    return model.params[0], model.params[1]


alpha, beta = linreg(X, Y)
print(f"Alpha: {alpha}")
print(f"Beta: {beta}")

Show the returns with the regression line.

In [ ]:
X2 = np.linspace(X.min(), X.max(), 100)
Y_hat = alpha + X2 * beta

plt.scatter(X, Y, alpha=0.3)
plt.plot(X2, Y_hat, "r", alpha=0.9)
plt.xlabel("SPY daily return")
plt.ylabel("Portfolio daily return")

Build a beta-hedged portfolio to neutralise market movements.

In [ ]:
hedged_portfolio_returns = -beta * benchmark_returns + portfolio_returns
hedged_portfolio_returns.name = "Hedged portfolio"

Regress the hedged portfolio: its beta is now about zero.

In [ ]:
P = hedged_portfolio_returns.values
alpha, beta = linreg(X, P)
print(f"Alpha: {alpha}")
print(f"Beta: {round(beta, 6)}")

In [ ]:
hedged_portfolio_returns.plot()
benchmark_returns.plot()
plt.ylabel("Daily Return")
plt.legend()